# 00 — Data Audit

Stage 2 (Data Understanding), structural checks. Confirms the raw panel is complete, well-typed, and within expected ranges before any feature engineering happens.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import config

pd.set_option("display.max_columns", 50)


## Load raw data

In [2]:
df = pd.read_csv(config.DATA_RAW, parse_dates=[config.DATE_COL])
print(df.shape)
df.head()


(73100, 15)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [3]:
df.dtypes

Date                  datetime64[ns]
Store ID                      object
Product ID                    object
Category                      object
Region                        object
Inventory Level                int64
Units Sold                     int64
Units Ordered                  int64
Demand Forecast              float64
Price                        float64
Discount                       int64
Weather Condition             object
Holiday/Promotion              int64
Competitor Pricing           float64
Seasonality                   object
dtype: object

## Panel completeness

Every (Store ID, Product ID) pair should have exactly one row per date, no gaps, no duplicates.

In [4]:
n_dates = df[config.DATE_COL].nunique()
n_series = df.groupby(config.ID_COLS).ngroups
expected_rows = n_dates * n_series
print(f"unique dates: {n_dates}, unique series: {n_series}, expected rows: {expected_rows}, actual rows: {len(df)}")
assert expected_rows == len(df), "panel is not a full dense grid"


unique dates: 731, unique series: 100, expected rows: 73100, actual rows: 73100


In [5]:
counts = df.groupby(config.ID_COLS).size()
print("rows per series - min/max:", counts.min(), counts.max())
assert (counts == n_dates).all(), "some series have missing/duplicate dates"

dup = df.duplicated(subset=config.ID_COLS + [config.DATE_COL]).sum()
print("duplicate (store, product, date) rows:", dup)
assert dup == 0


rows per series - min/max: 731 731
duplicate (store, product, date) rows: 0


## Missing values & dtypes

In [6]:
df.isna().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Demand Forecast       0
Price                 0
Discount              0
Weather Condition     0
Holiday/Promotion     0
Competitor Pricing    0
Seasonality           0
dtype: int64

## Range / sanity checks

In [7]:
checks = {
    "Inventory Level >= 0": (df["Inventory Level"] >= 0).all(),
    "Units Sold >= 0": (df["Units Sold"] >= 0).all(),
    "Units Ordered >= 0": (df["Units Ordered"] >= 0).all(),
    "Discount in [0,100]": df["Discount"].between(0, 100).all(),
    "Price > 0": (df["Price"] > 0).all(),
    "Competitor Pricing > 0": (df["Competitor Pricing"] > 0).all(),
    "Holiday/Promotion binary": set(df["Holiday/Promotion"].unique()) <= {0, 1},
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'FAIL'} - {k}")


OK  - Inventory Level >= 0
OK  - Units Sold >= 0
OK  - Units Ordered >= 0
OK  - Discount in [0,100]
OK  - Price > 0
OK  - Competitor Pricing > 0
OK  - Holiday/Promotion binary


## Categorical cardinalities

In [8]:
for col in ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]:
    print(col, "->", df[col].nunique(), sorted(df[col].unique())[:10])


Store ID -> 5 ['S001', 'S002', 'S003', 'S004', 'S005']
Product ID -> 20 ['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010']
Category -> 5 ['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']
Region -> 4 ['East', 'North', 'South', 'West']
Weather Condition -> 4 ['Cloudy', 'Rainy', 'Snowy', 'Sunny']
Seasonality -> 4 ['Autumn', 'Spring', 'Summer', 'Winter']


## Save cleaned panel

Cast dtypes (categoricals, int8 flags) and persist to `data/interim/cleaned_panel.parquet` for reuse by every downstream stage. Uses `src/data/clean.py` so the logic is shared with the pipeline, not duplicated here.

In [9]:
from src.data.clean import clean_panel

cleaned = clean_panel(df)
cleaned.dtypes


Date                  datetime64[ns]
Store ID                    category
Product ID                  category
Category                    category
Region                      category
Inventory Level                int64
Units Sold                     int64
Units Ordered                  int64
Demand Forecast              float64
Price                        float64
Discount                       int64
Weather Condition           category
Holiday/Promotion               int8
Competitor Pricing           float64
Seasonality                 category
dtype: object

In [10]:
config.DATA_INTERIM.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_parquet(config.DATA_INTERIM, index=False)
print("saved:", config.DATA_INTERIM, cleaned.shape)


saved: /Users/athens/Downloads/Coding/Retail Store Inventory/data/interim/cleaned_panel.parquet (73100, 15)


## Audit summary

- Panel is a full dense grid: 5 stores x 20 products x 731 days, no gaps or duplicates.
- No missing values in any column.
- All range checks pass (no negative inventory/sales, discount within [0,100], prices positive, promotion flag binary).
- Cleaned, dtype-cast panel saved to `data/interim/cleaned_panel.parquet` for Stage 3 (feature engineering).